# 01 – Data Overview
First look at the Credit Risk dataset: structure, data types, missing values,
duplicates, impossible values and the target balance.

In [3]:
import sys
from pathlib import Path

# The notebook runs from notebooks/, so add the project root to make `src` importable
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from src.config import TARGET, NUMERIC_COLS, CATEGORICAL_COLS
from src.data_loader import load_raw_data

pd.set_option("display.max_columns", None)

In [4]:
df = load_raw_data()
print(f"Rows: {df.shape[0]:,}  Columns: {df.shape[1]}")
df.head()

Rows: 32,581  Columns: 12


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


## Columns

| person_age - Borrower age |
| person_income - Annual income |
| person_home_ownership - RENT / OWN / MORTGAGE / OTHER |
| person_emp_length - Years employed |
| loan_intent - Purpose of the loan |
| loan_grade - Lender's risk grade, A (safest) to G |
| loan_amnt - Loan amount |
| loan_int_rate - Interest rate (%) |
| loan_status - **Target**: 1 = default, 0 = repaid |
| loan_percent_income - Loan amount ÷ income |
| cb_person_default_on_file - Defaulted before? Y/N (credit bureau) |
| cb_person_cred_hist_length - Years of credit history |

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  str    
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  str    
 5   loan_grade                  32581 non-null  str    
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  str    
 11  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(3), int64(5), str(4)
memory usage: 3.0 MB


In [6]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing": missing, "pct": missing_pct}).query("missing > 0")

,missing,pct
person_emp_length,895,2.75
loan_int_rate,3116,9.56


In [7]:
dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes}")

Duplicate rows: 165


In [9]:
df[NUMERIC_COLS].describe().T

,count,mean,std,min,25%,50%,75%,max
person_age,32581.0,27.734600,6.348078,20.00,23.00,26.00,30.00,144.00
person_income,32581.0,66074.848470,61983.119168,4000.00,38500.00,55000.00,79200.00,6000000.00
person_emp_length,31686.0,4.789686,4.142630,0.00,2.00,4.00,7.00,123.00
loan_amnt,32581.0,9589.371106,6322.086646,500.00,5000.00,8000.00,12200.00,35000.00
loan_int_rate,29465.0,11.011695,3.240459,5.42,7.90,10.99,13.47,23.22
loan_percent_income,32581.0,0.170203,0.106782,0.00,0.09,0.15,0.23,0.83
cb_person_cred_hist_length,32581.0,5.804211,4.055001,2.00,3.00,4.00,8.00,30.00


In [10]:
print("Age > 100:", (df["person_age"] > 100).sum())
print("Employment length > 60 years:", (df["person_emp_length"] > 60).sum())
print("Employed longer than alive:", (df["person_emp_length"] > df["person_age"]).sum())
df[(df["person_age"] > 100) | (df["person_emp_length"] > 60)]

Age > 100: 5
Employment length > 60 years: 2
Employed longer than alive: 2


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
81,144,250000,RENT,4.0,VENTURE,C,4800,13.57,0,0.02,N,3
183,144,200000,MORTGAGE,4.0,EDUCATION,B,6000,11.86,0,0.03,N,2
210,21,192000,MORTGAGE,123.0,VENTURE,A,20000,6.54,0,0.10,N,4
575,123,80004,RENT,2.0,EDUCATION,B,20400,10.25,0,0.25,N,3
747,123,78000,RENT,7.0,VENTURE,B,20000,NaN,0,0.26,N,4
32297,144,6000000,MORTGAGE,12.0,PERSONAL,C,5000,12.73,0,0.00,N,25


In [11]:
counts = df[TARGET].value_counts()
rates = df[TARGET].value_counts(normalize=True).mul(100).round(2)
pd.DataFrame({"count": counts, "pct": rates})

,count,pct
loan_status,,
0,25473,78.18
1,7108,21.82


In [12]:
for col in CATEGORICAL_COLS:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())


--- person_home_ownership ---
person_home_ownership
RENT        16446
MORTGAGE    13444
OWN          2584
OTHER         107
Name: count, dtype: int64

--- loan_intent ---
loan_intent
EDUCATION            6453
MEDICAL              6071
VENTURE              5719
PERSONAL             5521
DEBTCONSOLIDATION    5212
HOMEIMPROVEMENT      3605
Name: count, dtype: int64

--- loan_grade ---
loan_grade
A    10777
B    10451
C     6458
D     3626
E      964
F      241
G       64
Name: count, dtype: int64

--- cb_person_default_on_file ---
cb_person_default_on_file
N    26836
Y     5745
Name: count, dtype: int64


## Key findings
- 32,581 rows, 12 columns
- Missing: person_emp_length 2.75%, loan_int_rate 9.56% → impute in Phase 4
- 165 duplicate rows → drop
- Impossible values: 5 ages > 100, 2 employment lengths > 60 years → remove
- Default rate 21.82% → imbalanced; accuracy is misleading, use recall / F1 / ROC-AUC
- Rare category: person_home_ownership = OTHER (only ___ rows)